# Deployment with FastAPI

This notebook prepares the trained DeBERTa V3 model for deployment using
FastAPI.

The notebook performs the complete deployment pipeline including:

- Loading deployment artifacts
- Restoring the trained model
- Text preprocessing
- Top-3 prediction
- FastAPI application generation
- Deployment file generation

In [1]:
# Data Manipulation
import numpy as np
import pandas as pd

# Deep Learning
import torch
import torch.nn.functional as F

# Transformers
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

# Utilities
import joblib
import json
from pathlib import Path

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# Project Structure

try:

    from google.colab import drive

    drive.mount("/content/drive")

    BASE_DIR = Path("/content/drive/MyDrive/Mental_Health_Classification")

except:

    print("Running outside Google Colab...")

    BASE_DIR = Path.cwd()

PROJECT_DIR = BASE_DIR
DATA_DIR = PROJECT_DIR / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
MODELS_DIR = PROJECT_DIR / "models"
TOKENIZERS_DIR = PROJECT_DIR / "tokenizers"
ENCODERS_DIR = PROJECT_DIR / "encoders"
RESULTS_DIR = PROJECT_DIR / "results"
FIGURES_DIR = PROJECT_DIR / "figures"
NOTEBOOKS_DIR = PROJECT_DIR / "notebooks"
DEPLOYMENT_DIR = PROJECT_DIR / "deployment"

print("Project directories loaded successfully.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project directories loaded successfully.


In [3]:
# Load Deployment Assets

MODEL_PATH = DEPLOYMENT_DIR / "best_deberta_v3.pt"
TOKENIZER_PATH = DEPLOYMENT_DIR / "tokenizer"
LABEL_ENCODER_PATH = DEPLOYMENT_DIR / "label_encoder.pkl"
CONFIG_PATH = DEPLOYMENT_DIR / "config.json"

In [4]:
with open(

    CONFIG_PATH,
    encoding="utf-8"

) as f:
    config = json.load(f)

label_encoder = joblib.load(
    LABEL_ENCODER_PATH
)

CLASS_NAMES = config["class_names"]
MAX_LENGTH = config["max_length"]
MODEL_NAME = config["model_name"]

print("Deployment assets loaded successfully.")

Deployment assets loaded successfully.


In [5]:
# Load Tokenizer

tokenizer = AutoTokenizer.from_pretrained(
    TOKENIZER_PATH
)
print("Tokenizer loaded.")

# Load Model

device = torch.device(

    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = AutoModelForSequenceClassification.from_pretrained(

    MODEL_NAME,
    num_labels=len(CLASS_NAMES)
)
state_dict = torch.load(

    MODEL_PATH,
    map_location=device
)
model.load_state_dict(
    state_dict
)
model.to(device)
model.eval()

print("Model loaded successfully.")

Tokenizer loaded.


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.den

Model loaded successfully.


In [6]:
# preprocessing

import sys

sys.path.append(str(PROJECT_DIR))

from utils.preprocessing import preprocess_text

print("Preprocessing pipeline loaded successfully.")

Preprocessing pipeline loaded successfully.


In [16]:
# Prediction Function

def predict(text, top_k=3):

    if not isinstance(text, str):

        raise ValueError("Input must be a string.")

    if text.strip() == "":

        raise ValueError("Input text is empty.")

    text = preprocess_text(

        text,
        mode="transformer"
    )

    encoded = tokenizer(

        text,
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )

    input_ids = encoded["input_ids"].to(device)
    attention_mask = encoded["attention_mask"].to(device)

    with torch.no_grad():

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

    probabilities = F.softmax(
        outputs.logits,
        dim=1
    )[0]

    values, indices = torch.topk(
        probabilities,
        k=top_k
    )

    predictions = []

    for score, idx in zip(values, indices):

        predictions.append({

            "rank": len(predictions)+1,
            "label": CLASS_NAMES[idx.item()],
            "confidence": round(score.item()*100,2)
            })

    return {

        "prediction": predictions[0]["label"],
        "confidence": predictions[0]["confidence"],
        "top3": predictions
    }

In [17]:
sample = """
I feel hopeless and empty.
Nothing makes me happy anymore.
"""

prediction = predict(sample)

print("=" * 60)
print("Predicted Class")
print("=" * 60)
print(
    prediction["prediction"]
)
print()
print("Confidence")
print(
    f'{prediction["confidence"]:.2f}%'
)
print()

top3_df = pd.DataFrame(
    prediction["top3"]
)

top3_df

Predicted Class
Depression

Confidence
93.55%



,rank,label,confidence
0,1,Depression,93.55
1,2,Suicidal,6.32
2,3,Stress,0.05


In [18]:
def batch_predict(texts):

    results = []

    for text in texts:

        pred = predict(text)
        results.append({

            "Text": text,
            "Prediction": pred["prediction"],
            "Confidence": pred["confidence"],
            "Top 3":pred["top3"]

        })

    return pd.DataFrame(results)

In [27]:
examples = [

    "Everything is amazing today!",
    "I don't want to live anymore."

]

batch_predict(examples)

,Text,Prediction,Confidence,Top 3
0,Everything is amazing today!,Normal,100.00,"[{'rank': 1, 'label': 'Normal', 'confidence': ..."
1,I don't want to live anymore.,Suicidal,81.59,"[{'rank': 1, 'label': 'Suicidal', 'confidence'..."


# Generate FastAPI App

In [20]:
from textwrap import dedent

In [21]:
app_code = dedent("""

from fastapi import FastAPI
from pydantic import BaseModel
from inference import predict

app = FastAPI(

    title="Mental Health Classification API",
    version="1.0.0"
)

class InputText(BaseModel):

    text: str


@app.get("/")
def home():

    return {

        "message":"Mental Health Classification API"
    }


@app.post(

    "/predict",
    tags=["Prediction"]
)

def classify(data: InputText):

    return predict(data.text)

""")

In [22]:
with open(
    DEPLOYMENT_DIR / "app.py",
    "w",
    encoding="utf-8"

) as f:
    f.write(app_code)

print("app.py created successfully.")

app.py created successfully.


# Generate inference.py

In [23]:
import shutil

shutil.copy(

    PROJECT_DIR / "utils" / "preprocessing.py",
    DEPLOYMENT_DIR / "preprocessing.py"
)

print("Preprocessing copied successfully.")

Preprocessing copied successfully.


In [32]:
inference_code = dedent("""

import json
import torch
import torch.nn.functional as F
import joblib
from pathlib import Path
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification
from preprocessing import preprocess_text

BASE_DIR = Path(__file__).resolve().parent
MODEL_PATH = BASE_DIR / "best_deberta_v3.pt"
TOKENIZER_PATH = BASE_DIR / "tokenizer"
LABEL_ENCODER_PATH = BASE_DIR / "label_encoder.pkl"
CONFIG_PATH = BASE_DIR / "config.json"

with open(CONFIG_PATH, encoding="utf-8") as f:

    config = json.load(f)

label_encoder = joblib.load(

    LABEL_ENCODER_PATH
)

CLASS_NAMES = config["class_names"]
MAX_LENGTH = config["max_length"]
MODEL_NAME = config["model_name"]

device = torch.device(

    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

tokenizer = AutoTokenizer.from_pretrained(

    TOKENIZER_PATH
)

model = AutoModelForSequenceClassification.from_pretrained(

    MODEL_NAME,
    num_labels=len(CLASS_NAMES)
)

model.load_state_dict(

    state_dict = torch.load(

        MODEL_PATH,
        map_location=device,
        weights_only=True
    )
)

model.to(device)
model.eval()
torch.set_grad_enabled(False)

def predict(text):

    text = preprocess_text(

        text,
        mode="transformer"
    )

    encoded = tokenizer(

        text,
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )

    input_ids = encoded["input_ids"].to(device)
    attention_mask = encoded["attention_mask"].to(device)

    with torch.no_grad():

        logits = model(

            input_ids=input_ids,
            attention_mask=attention_mask
        ).logits


    probs = F.softmax(

        logits,
        dim=1
    )[0]


    values, indices = torch.topk(

        probs,
        k=3
    )

    top3 = []

    for score, idx in zip(values, indices):

        top3.append({

            "label": CLASS_NAMES[idx.item()],
            "confidence": round(

                score.item()*100,
                2
            )
        })

    return {

        "prediction": top3[0]["label"],
        "confidence": top3[0]["confidence"],
        "top3": top3
    }

""")

In [33]:
with open(
    DEPLOYMENT_DIR / "inference.py",
    "w",
    encoding="utf-8"

) as f:
    f.write(inference_code)

print("inference.py created successfully.")

inference.py created successfully.


In [34]:
requirements = """
fastapi
uvicorn
torch
transformers
joblib
numpy
pandas
scikit-learn
nltk
sentencepiece
accelerate
"""

with open(
    DEPLOYMENT_DIR / "requirements.txt",
    "w",
    encoding="utf-8"
) as f:

    f.write(requirements.strip())

print("requirements.txt created successfully.")

requirements.txt created successfully.


In [35]:
readme = """
# Mental Health Classification API

## Run

uvicorn app:app --reload

## Endpoint

POST /predict

Body

{
    "text":"I feel hopeless."
}

Response

{
    "prediction":"Depression",
    "confidence":92.51,
    "top3":[]
}
"""

with open(
    DEPLOYMENT_DIR/"README.md",
    "w",
    encoding="utf-8"
) as f:

    f.write(readme)

print("README created successfully.")

README created successfully.


In [36]:
print("="*60)
print("Deployment Files")
print("="*60)

for file in sorted(
    DEPLOYMENT_DIR.iterdir()
):

    print(file.name)

Deployment Files
README.md
app.py
best_deberta_v3.pt
config.json
inference.py
label_encoder.pkl
preprocessing.py
requirements.txt
tokenizer


In [37]:
print("="*60)
print("Example API Response")
print("="*60)

response = predict(

    "I have been feeling empty and hopeless lately."
)

response

Example API Response


{'prediction': 'Depression',
 'confidence': 75.93,
 'top3': [{'rank': 1, 'label': 'Depression', 'confidence': 75.93},
  {'rank': 2, 'label': 'Suicidal', 'confidence': 23.99},
  {'rank': 3, 'label': 'Stress', 'confidence': 0.03}]}